# Projeto Prático 1 (Parte 1) — Aproximação de Funções com `PMC` em `NumPy`

> ** Contexto de Engenharia Biomédica / Elétrica (Seção 5.8 do Livro-Base):**
> Em um tomógrafo de **Ressonância Magnética**, deseja-se estimar continuamente a **energia absorvida** ($d \in [0, 1]$) pelo sistema a partir de três grandezas medidas em tempo real ($x_1, x_2, x_3$). Como o fenômeno eletromagnético é fortemente não linear, utilizamos um **Perceptron Multicamadas (PMC)** como **aproximador universal de funções**!

<div align="center">
    <img src="../imagens/topologia_pmc_3_10_1.svg" alt="Topologia 3-10-1 do PMC" width="560">
</div>

### Especificações de Projeto
- **Topologia:** `3-10-1` (3 entradas, 10 neurônios ocultos, 1 neurônio de saída contínua)
- **Ativação:** Logística / Sigmóide $\sigma(u) = \frac{1}{1 + e^{-u}}$ em todas as camadas
- **Taxa de aprendizado ($\eta$):** `0,1` | **Precisão ($\varepsilon$):** $10^{-6}$ (`MSE`) | **Limite:** `5000` épocas
- **Parte 2 (Industrial):** Veja a comparação com `Scikit-Learn` e `Keras` em [`03_projeto1_comparacao_sklearn_keras.ipynb`](03_projeto1_comparacao_sklearn_keras.ipynb).

---

## 1. Carregamento Limpo dos Conjuntos de Treinamento e Teste

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline

# Lê diretamente os arquivos CSV padronizados da pasta datasets/
train_df = pd.read_csv("datasets/projeto-pratico-1.csv")
test_df = pd.read_csv("datasets/conjunto-testes-projeto-pratico-1.csv")

print(f"Amostras de treinamento: {len(train_df)} | Amostras de teste: {len(test_df)}")
print("\nPrimeiras 5 linhas do Treinamento:")
print(train_df.head())
print("\nPrimeiras 5 linhas do Teste:")
print(test_df.head())

**Observação:** Tanto entradas ($x_1, x_2, x_3$) quanto a saída desejada ($d$) já estão normalizadas no intervalo `[0, 1]`, compatível com a imagem da função logística sigmóide!

---

## 2. Implementação Didática da Classe `MLPRegression` (Linha a Linha)

Observe atentamente as dimensões matriciais de cada operação para uma rede `3-10-1`:
- **`self.W1` (`10 x 4`):** Conecta `[x0=-1, x1, x2, x3]` (`4 x 1`) aos `10` neurônios ocultos (`10 x 1`).
- **`self.W2` (`1 x 11`):** Conecta `[y0=-1, y1_1, ..., y1_10]` (`11 x 1`) ao único neurônio de saída (`1 x 1`).
- Na retropropagação (`Backward`), usamos `self.W2[:, 1:]` (`1 x 10`) porque o fio fixo do *bias* (`-1`) não recebe erro de volta!

In [ ]:
class MLPRegression:
    def __init__(self, input_size, hidden_size, output_size, seed=42):
        # Gerador isolado com semente específica para cada um dos 5 treinamentos
        self.rng = np.random.default_rng(seed)

        # Inicializa matrizes de pesos uniformemente entre -0.5 e +0.5 (incluindo coluna 0 para bias)
        self.W1 = self.rng.uniform(-0.5, 0.5, size=(hidden_size, input_size + 1))
        self.W2 = self.rng.uniform(-0.5, 0.5, size=(output_size, hidden_size + 1))

    @staticmethod
    def sigmoid(u):
        # Função logística contínua entre 0 e 1
        return 1.0 / (1.0 + np.exp(-u))

    @staticmethod
    def sigmoid_derivative(y):
        # Derivada da sigmóide expressa em função da própria saída y: g'(u) = y * (1 - y)
        return y * (1.0 - y)

    def train(self, X, D, eta=0.1, epsilon=1e-6, max_epochs=5000):
        N = X.shape[0]          # Número total de ensaios de treinamento (200)
        mse_history = []        # Histórico do Erro Quadrático Médio por época

        for epoch in range(1, max_epochs + 1):
            soma_erro_quadratico = 0.0

            # Embaralha a ordem de apresentação das amostras a cada época (evita ciclos viciosos)
            indices = self.rng.permutation(N)

            for idx in indices:
                # Transforma linha em vetor-coluna (3x1) e (1x1)
                x = X[idx].reshape(-1, 1)
                d = D[idx].reshape(-1, 1)

                # --- 1. PROPAGAÇÃO DIRETA (Forward Pass) ---
                x_bias = np.vstack([[-1.0], x])             # (4x1): empilha -1 no topo
                y1 = self.sigmoid(self.W1 @ x_bias)         # (10x1): ativação da camada oculta

                y1_bias = np.vstack([[-1.0], y1])           # (11x1): empilha -1 no topo de y1
                y2 = self.sigmoid(self.W2 @ y1_bias)        # (1x1): estimativa contínua de saída

                # --- 2. CÁLCULO DO ERRO INSTANTÂNEO ---
                erro = d - y2                               # Diferença escalar (1x1)
                soma_erro_quadratico += 0.5 * float(np.sum(erro ** 2))

                # --- 3. RETROPROPAGAÇÃO (Backward Pass / Regra da Cadeia) ---
                delta2 = erro * self.sigmoid_derivative(y2) # Gradiente local da saída (1x1)
                # Propaga delta2 para os 10 neurônios ocultos ignorando a coluna 0 (bias de W2)
                delta1 = self.sigmoid_derivative(y1) * (self.W2[:, 1:].T @ delta2)

                # --- 4. AJUSTE ONLINE DOS PESOS (Regra Delta Generalizada) ---
                self.W2 = self.W2 + eta * (delta2 @ y1_bias.T)
                self.W1 = self.W1 + eta * (delta1 @ x_bias.T)

            # Erro Quadrático Médio da época completa
            mse = soma_erro_quadratico / N
            mse_history.append(mse)

            # Critério de parada por precisão atingida
            if mse <= epsilon:
                return mse_history, epoch

        return mse_history, max_epochs

    def predict(self, X):
        # Executa apenas a viagem de ida (Forward Pass) para novas amostras de teste
        predicoes = []
        for i in range(X.shape[0]):
            x_bias = np.vstack([[-1.0], X[i].reshape(-1, 1)])
            y1 = self.sigmoid(self.W1 @ x_bias)
            y1_bias = np.vstack([[-1.0], y1])
            y2 = self.sigmoid(self.W2 @ y1_bias)
            predicoes.append(float(y2[0, 0]))
        return np.array(predicoes)

---

## 3. Executando os Cinco Treinamentos (`T1` a `T5`) — Tabela 5.2

Usamos uma lista fixa de 5 sementes distintas (`[101, 202, 303, 404, 505]`) para garantir **dois requisitos simultâneos**: pesos iniciais completamente diferentes entre `T1..T5` e **100% de reprodutibilidade científica** toda vez que você rodar este notebook!

In [ ]:
X_train = train_df[["x1", "x2", "x3"]].to_numpy()
D_train = train_df[["d"]].to_numpy()

sementes = [101, 202, 303, 404, 505]
resultados_t = []
modelos = []
historicos = []

for i, semente in enumerate(sementes):
    rede = MLPRegression(input_size=3, hidden_size=10, output_size=1, seed=semente)
    hist_mse, epocas_gastas = rede.train(X_train, D_train, eta=0.1, epsilon=1e-6, max_epochs=5000)

    modelos.append(rede)
    historicos.append(hist_mse)
    resultados_t.append({
        "Treinamento": f"T{i + 1}",
        "Semente": semente,
        "Erro Quadrático Médio (MSE)": hist_mse[-1],
        "Total de Épocas": epocas_gastas,
    })

df_tabela_52 = pd.DataFrame(resultados_t)
print("Tabela 5.2 – Resultados dos 5 Treinamentos:")
print(df_tabela_52.to_string(index=False))

---

## 4. Curvas de Convergência e Explicação Física da Variabilidade

Conforme solicitado no projeto:
1. Plotamos em **subgráficos separados (não superpostos)** os dois treinamentos com maior número de épocas;
2. Em seguida, comparamos todos os 5 treinamentos em escala logarítmica para inspecionar a velocidade fina de descida do gradiente.

In [ ]:
# 1. Seleciona os dois treinamentos mais longos (em caso de empate em 5000, pega os 2 maiores MSE)
indices_ordenados = df_tabela_52.sort_values(
    by=["Total de Épocas", "Erro Quadrático Médio (MSE)"], ascending=[False, False]
).index[:2]

fig, axes = plt.subplots(2, 1, figsize=(9, 7))
for idx, ax in zip(indices_ordenados, axes):
    nome_t = df_tabela_52.loc[idx, "Treinamento"]
    ep_t = df_tabela_52.loc[idx, "Total de Épocas"]
    ax.plot(historicos[idx], color="royalblue", linewidth=1.8)
    ax.set_title(f"Curva de Erro — Treinamento {nome_t} ({ep_t} épocas)")
    ax.set_xlabel("Época")
    ax.set_ylabel("MSE")
    ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

# 2. Comparação conjunta em escala logarítmica (evidencia diferenças de 10^-4 vs 10^-5)
plt.figure(figsize=(9, 4.5))
for i, hist in enumerate(historicos):
    plt.plot(hist, label=f"Treinamento T{i + 1}")
plt.yscale("log")
plt.title("Comparação Logarítmica do MSE nos 5 Treinamentos")
plt.xlabel("Época")
plt.ylabel("MSE (escala log)")
plt.legend()
plt.grid(True, which="both", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

### Por que o Erro Final varia entre `T1` e `T5`?
Em uma rede multicamadas, a superfície de custo $E(\mathbf{W})$ possui relevo **não convexo** (como uma cadeia de montanhas cheia de vales secundários e platôs planos). Cada semente aleatória solta a "esfera" do Gradiente Descendente em uma encosta diferente: algumas descem direto para um vale mais fundo (menor MSE), enquanto outras atravessam regiões quase planas (*pontos de sela*) onde as derivadas são pequenas.

---

## 5. Validação em Amostras Inéditas de Teste (Tabela 5.3) e Escolha do Melhor Modelo

Avaliamos agora os 5 modelos sobre as **20 amostras inéditas** de `datasets/conjunto-testes-projeto-pratico-1.csv`, calculando:
$$\text{Erro Relativo (\%)} = \frac{|d - y|}{|d|} \times 100\%$$

In [ ]:
X_test = test_df[["x1", "x2", "x3"]].to_numpy()
D_test = test_df["d"].to_numpy()

resumo_validacao = []
tabela_detalhada = test_df.copy()

for i, rede in enumerate(modelos):
    y_pred = rede.predict(X_test)
    tabela_detalhada[f"y_T{i + 1}"] = np.round(y_pred, 4)

    # Erro relativo percentual em cada uma das 20 amostras de teste
    erros_relativos_pct = (np.abs(D_test - y_pred) / D_test) * 100.0

    resumo_validacao.append({
        "Treinamento": f"T{i + 1}",
        "Erro Relativo Médio (%)": np.mean(erros_relativos_pct),
        "Variância do Erro (%)": np.var(erros_relativos_pct),
    })

df_tabela_53 = pd.DataFrame(resumo_validacao)
melhor_idx = df_tabela_53["Erro Relativo Médio (%)"].idxmin()
melhor_treino = df_tabela_53.loc[melhor_idx, "Treinamento"]

print("Primeiras 8 amostras previstas (Desejado d vs. Redes T1..T5):")
print(tabela_detalhada.head(8).to_string(index=False))

print("\nTabela 5.3 – Métricas de Generalização no Conjunto de Teste:")
print(df_tabela_53.round(4).to_string(index=False))
print(f"\n>>> Melhor configuração para embarcar no tomógrafo: {melhor_treino} <<<")

**Análise de Engenharia:** Com erro relativo médio na casa de **~1,6% a 1,9%**, a rede `3-10-1` estima com altíssima fidelidade a energia absorvida no equipamento de ressonância magnética!

---

## 6. Exercícios Práticos de Fixação

### Exercício 1: Gráfico Real vs. Previsto do Melhor Modelo
Plote um gráfico de dispersão (`plt.scatter(D_test, y_melhor)`) comparando os valores reais `D_test` (eixo X) com as estimativas do melhor modelo (eixo Y), junto com a reta ideal de $45^\circ$ (`plt.plot([0.2, 0.9], [0.2, 0.9], 'r--')`). Quanto mais colados na linha tracejada, mais preciso é o sensor virtual!

In [ ]:
# Escreva seu código aqui

<details>
<summary><b>Clique aqui para ver o gabarito comentado</b></summary>

```python
rede_campeao = modelos[melhor_idx]
y_melhor = rede_campeao.predict(X_test)

plt.figure(figsize=(6, 6))
plt.scatter(D_test, y_melhor, color="darkblue", s=60, label=f"Predições ({melhor_treino})")
plt.plot([0.2, 0.9], [0.2, 0.9], "r--", linewidth=2, label="Ajuste Perfeito (y = d)")
plt.title("Energia Absorvida: Medida Real vs. Estimativa da RNA")
plt.xlabel("Valor Real Desejado (d)")
plt.ylabel("Valor Estimado pela Rede (y)")
plt.legend()
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()
```

</details>

### Exercício 2: Efeito do Número de Neurônios Ocultos (Subajuste vs. Capacidade)
Treine uma rede minúscula com apenas **2 neurônios ocultos** (`3-2-1`, `max_epochs=2000`, `seed=101`) e compare seu **Erro Relativo Médio (%)** no teste contra uma rede `3-10-1` treinada por `2000` épocas.

In [ ]:
# Escreva seu código aqui

<details>
<summary><b>Clique aqui para ver o gabarito comentado</b></summary>

```python
for n_ocultos in [2, 10]:
    r_teste = MLPRegression(3, n_ocultos, 1, seed=101)
    r_teste.train(X_train, D_train, eta=0.1, max_epochs=2000)
    pred = r_teste.predict(X_test)
    erro_rel = np.mean(np.abs(D_test - pred) / D_test * 100.0)
    print(f"Topologia 3-{n_ocultos}-1 (2000 épocas) -> Erro Relativo Médio no Teste: {erro_rel:.3f}%")
```

</details>

---

## Próximo Passo
Veja agora como resolver este exato problema de Ressonância Magnética em poucas linhas com bibliotecas industriais em **[`03_projeto1_comparacao_sklearn_keras.ipynb`](03_projeto1_comparacao_sklearn_keras.ipynb)**!